> ## SOLUTION / ANSWER KEY &mdash; Lab 6.5
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-05-run-and-trace.ipynb`](../lab-05-run-and-trace.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.5 &mdash; Run the Agent & Read the Trace

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Run a real agent and capture the list of messages it returns
- Extract which tools were called (from each AIMessage.tool_calls)
- Cap the loop with recursion_limit so it can never run forever

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; The agent loop, made visible](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-05"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
`create_agent` runs the loop for you: the model decides, tools run, results feed back &mdash; until a
final answer, or until **`recursion_limit`** stops it. The agent returns a **list of messages** (the
trace): **`AIMessage`** with **`.tool_calls`** (what it decided to run), **`ToolMessage`** (the result),
and a final `AIMessage` with **`.content`**. Here you run a real agent and read *its* trace.

In [ ]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search for a current fact. Use for figures the model may not know."""
    return {"population of france": "68000000"}.get(query.lower().strip(), "no result")

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic, e.g. '68000000/2'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return "error: not a numeric expression"

print("two tools ready:", web_search.name, "&", calculator.name)

## Build it
Complete `run_config` (the loop cap the agent honors), `tools_used` (every tool call, in order), and
`final_answer` (the last message with text). You'll apply them to a **real** trace next.

In [ ]:
from langchain_core.messages import AIMessage
from langchain.agents import create_agent

def run_config(max_steps):
    return {"recursion_limit": max_steps}

def tools_used(messages):
    names = []
    for m in messages:
        for tc in (getattr(m, "tool_calls", None) or []):   # AIMessages carry tool_calls
            names.append(tc["name"])
    return names

def final_answer(messages):
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content:   # the last AIMessage that has text
            return m.content
    return None

agent = create_agent(llm, tools=[web_search, calculator])

try:
    print("run config:", run_config(8))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run a real compound query, then read the trace with YOUR functions: which tools fired, and the final answer.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        result = agent.invoke(
            {"messages": [("user", "Use web_search for the population of France, then halve it with the calculator.")]},
            config=run_config(8))
        print("full trace:")
        print_trace(result)
        print("---")
        print("tools used  :", tools_used(result["messages"]))
        print("final answer:", final_answer(result["messages"]))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- **`tools_used`** reads straight off the real trace &mdash; it's your #1 debugging surface.
- **`recursion_limit`** is a one-line guardrail: even a confused agent can't loop forever.
- If the model skipped a tool or chained them differently, that's real behaviour &mdash; the trace tells you the truth.

## Your turn (open task &mdash; no grader)
Drop the cap to `run_config(2)` and re-run. **What good looks like:** on a task that needs two tool
steps, a cap of 2 can cut the agent off before it finishes &mdash; you'll see it in the trace. Raise the cap
and it completes. That's the safety/completeness trade-off you tune per task.

---
### Key takeaway
The agent's loop is now visible: tool_calls show what it did; recursion_limit is your one-line guardrail. The message trace is the truth of what happened.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>